# Fenway Park Game & Weather Dataset (2016-2025)

This notebook builds a comprehensive dataset where each row represents a regular season game played at Fenway Park. It combines:
1. **Game statistics** from `master_data.csv` (pre-aggregated Statcast data)
2. **Weather data** from Open-Meteo hourly observations
3. **Wind projections** onto outfield vectors (CF, LCF, RCF)

All batting/pitching statistics are **both teams combined** to capture the full park-environment effect.

## Section 0: Setup & Configuration

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os
import requests
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

# === STADIUM CONFIGURATION ===
STADIUM_NAME = 'Fenway Park'
HOME_TEAM = 'BOS'
SEASONS = range(2016, 2026)  # 2016 through 2025
TIMEZONE = 'America/New_York'

# Coordinates
STADIUM_LAT = 42.346268
STADIUM_LON = -71.095764

# Outfield directions (degrees from north)
CF_DIR = 45.0    # Center field: NE
LCF_DIR = 25.0   # Left-center field: NNE
RCF_DIR = 65.0   # Right-center field: ENE

# Output file
OUTPUT_FILE = 'fenway_data_2016.csv'

print(f"Configuration: {STADIUM_NAME}")
print(f"Home team: {HOME_TEAM}")
print(f"Seasons: {list(SEASONS)}")
print(f"Timezone: {TIMEZONE}")

/Users/avabrown/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/avabrown/opt/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.4' currently installed).
  from pandas.core import (


Configuration: Fenway Park
Home team: BOS
Seasons: [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
Timezone: America/New_York


## Section 1: Load & Filter Master Data

Read game-level statistics from `master_data.csv` and filter to Fenway Park home games.

In [2]:
# Load master dataset
master = pd.read_csv(os.path.join('..', 'Final Datasets', 'master_data.csv'))
print(f"Master dataset: {len(master)} total games")

# Filter to this stadium's home games and season range
games = master[
    (master['home_team'] == HOME_TEAM) &
    (master['season'].isin(SEASONS))
].copy()

# Convert game_start_utc to local timezone
games['game_start'] = (
    pd.to_datetime(games['game_start_utc'], utc=True)
    .dt.tz_convert(TIMEZONE)
    .dt.tz_localize(None)  # Remove timezone info for clean processing
)
games['start_hour'] = games['game_start'].dt.hour
games['game_date'] = pd.to_datetime(games['game_date'])

# Drop master-only columns not needed in final output
games = games.drop(columns=['home_team', 'game_start_utc'])

games = games.sort_values('game_date').reset_index(drop=True)

print(f"\n{STADIUM_NAME} games: {len(games)}")
print(f"Seasons: {sorted(games['season'].unique())}")
print(f"\nGames per season:")
print(games.groupby('season')['game_pk'].count())
print(f"\nSample:")
print(games.head(3))

Master dataset: 25155 total games

Fenway Park games: 758
Seasons: [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

Games per season:
season
2016    80
2017    81
2018    81
2019    81
2020    30
2021    81
2022    81
2023    81
2024    81
2025    81
Name: game_pk, dtype: int64

Sample:
   game_pk  game_date  season away_team  home_runs_scored  away_runs_scored  total_runs  home_runs_hit  strikeouts  walks  hits  total_pitches  avg_exit_velocity  n_barrels  n_bbe  barrel_rate  \
0   446958 2016-04-11    2016       BAL                 7                 9          16              3          23      8    20            330               84.0          6     49       0.1224   
1   446970 2016-04-12    2016       BAL                 5                 9          14              4          13      6    21            332               83.7          3     59       0.0508   
2   446983 2016-04-13    2016       BAL                 4                 2           6              1         

## Section 2: Pull Weather Data (Open-Meteo)

Use Open-Meteo to pull hourly weather data for each season, then average over the 3 hours following each game's start time.

In [3]:
# Using Open-Meteo Historical Weather API (free, no key required)
OPEN_METEO_URL = "https://archive-api.open-meteo.com/v1/archive"
HOURLY_PARAMS = "temperature_2m,relative_humidity_2m,surface_pressure,precipitation,wind_speed_10m,wind_direction_10m"

# Test fetch to confirm API is reachable
test_resp = requests.get(OPEN_METEO_URL, params={
    'latitude': STADIUM_LAT,
    'longitude': STADIUM_LON,
    'start_date': '2023-07-01',
    'end_date': '2023-07-02',
    'hourly': HOURLY_PARAMS,
    'timezone': TIMEZONE,
})

if test_resp.status_code == 200:
    test_data = test_resp.json()
    n_hours = len(test_data['hourly']['time'])
    print(f"Open-Meteo API test (Jul 1-2 2023): {n_hours} hourly records - OK")
    print(f"Sample time: {test_data['hourly']['time'][12]}")
    print(f"Sample temp: {test_data['hourly']['temperature_2m'][12]}°C")
    print(f"Sample wind: {test_data['hourly']['wind_speed_10m'][12]} km/h from {test_data['hourly']['wind_direction_10m'][12]}°")
else:
    print(f"ERROR: Open-Meteo API returned {test_resp.status_code}")
    print(test_resp.text)

Open-Meteo API test (Jul 1-2 2023): 48 hourly records - OK
Sample time: 2023-07-01T12:00
Sample temp: 23.5°C
Sample wind: 13.9 km/h from 80°


In [4]:
def fetch_season_weather(year):
    """Fetch hourly weather for a full season from Open-Meteo."""
    resp = requests.get(OPEN_METEO_URL, params={
        'latitude': STADIUM_LAT,
        'longitude': STADIUM_LON,
        'start_date': f'{year}-03-01',
        'end_date': f'{year}-11-30',
        'hourly': HOURLY_PARAMS,
        'timezone': TIMEZONE,
    })
    resp.raise_for_status()
    hourly = resp.json()['hourly']
    
    df = pd.DataFrame({
        'temp': hourly['temperature_2m'],
        'rhum': hourly['relative_humidity_2m'],
        'pres': hourly['surface_pressure'],
        'prcp': hourly['precipitation'],
        'wspd': hourly['wind_speed_10m'],
        'wdir': hourly['wind_direction_10m'],
    }, index=pd.to_datetime(hourly['time']))
    
    return df


def get_game_weather(game_start_dt, hourly_df):
    """
    Average weather over the 3 hours following game start.
    game_start_dt: datetime (local time, rounded to hour)
    hourly_df: DataFrame with hourly weather, index is naive local time
    """
    start = game_start_dt
    end = start + timedelta(hours=2)  # 3 hourly obs: start, +1h, +2h
    
    window = hourly_df.loc[start:end]
    
    if len(window) == 0:
        return pd.Series({
            'temp_c': np.nan, 'rhum': np.nan, 'pres': np.nan,
            'prcp': np.nan, 'wspd': np.nan, 'wdir': np.nan
        })
    
    result = {
        'temp_c': window['temp'].mean(),
        'rhum': window['rhum'].mean(),
        'pres': window['pres'].mean(),
        'prcp': window['prcp'].sum(),   # Precipitation SUMMED (cumulative quantity)
        'wspd': window['wspd'].mean(),
    }
    
    # Wind direction: circular mean to handle 0/360 boundary
    wdir_vals = window['wdir'].dropna()
    if len(wdir_vals) > 0:
        wdir_rad = np.radians(wdir_vals)
        mean_sin = np.sin(wdir_rad).mean()
        mean_cos = np.cos(wdir_rad).mean()
        result['wdir'] = np.degrees(np.arctan2(mean_sin, mean_cos)) % 360
    else:
        result['wdir'] = np.nan
    
    return pd.Series(result)


# Pull weather season by season
weather_records = []

for year in SEASONS:
    print(f"Pulling weather for {year}...")
    
    try:
        hourly_df = fetch_season_weather(year)
    except Exception as e:
        print(f"  WARNING: Failed for {year}: {e}")
        season_games = games[games['season'] == year]
        for idx, game in season_games.iterrows():
            weather_records.append({
                'game_pk': game['game_pk'],
                'temp_c': np.nan, 'rhum': np.nan, 'pres': np.nan,
                'prcp': np.nan, 'wspd': np.nan, 'wdir': np.nan
            })
        continue
    
    print(f"  {year}: {len(hourly_df)} hourly records")
    
    season_games = games[games['season'] == year]
    for idx, game in season_games.iterrows():
        if pd.isna(game['game_start']):
            weather_records.append({
                'game_pk': game['game_pk'],
                'temp_c': np.nan, 'rhum': np.nan, 'pres': np.nan,
                'prcp': np.nan, 'wspd': np.nan, 'wdir': np.nan
            })
            continue
        
        game_hour = game['game_start'].replace(minute=0, second=0, microsecond=0)
        wx = get_game_weather(game_hour, hourly_df)
        wx['game_pk'] = game['game_pk']
        weather_records.append(wx.to_dict())

weather_df = pd.DataFrame(weather_records)
weather_df['game_pk'] = weather_df['game_pk'].astype('Int64')
print(f"\nWeather records: {len(weather_df)}")
print(f"Missing temp data: {weather_df['temp_c'].isna().sum()}")
print(weather_df.head(3))

Pulling weather for 2016...
  2016: 6600 hourly records
Pulling weather for 2017...
  2017: 6600 hourly records
Pulling weather for 2018...
  2018: 6600 hourly records
Pulling weather for 2019...
  2019: 6600 hourly records
Pulling weather for 2020...
  2020: 6600 hourly records
Pulling weather for 2021...
  2021: 6600 hourly records
Pulling weather for 2022...
  2022: 6600 hourly records
Pulling weather for 2023...
  2023: 6600 hourly records
Pulling weather for 2024...
  2024: 6600 hourly records
Pulling weather for 2025...
  2025: 6600 hourly records

Weather records: 758
Missing temp data: 0
      temp_c       rhum         pres  prcp       wspd        wdir  game_pk
0  13.300000  52.000000  1018.033333   0.0  27.066667  196.333330   446958
1   6.700000  72.666667  1021.166667   0.0  16.933333  311.338055   446970
2   4.466667  75.000000  1026.933333   0.0   7.366667  129.000000   446983


## Section 3: Wind Direction Bucketing & Outfield Projections

**Wind direction bucketing**: 8 compass directions (N, NE, E, SE, S, SW, W, NW).

**Wind projections**: Project wind onto vectors from home plate to center field (CF), left-center field (LCF), and right-center field (RCF). Positive = blowing out, negative = blowing in.

Fenway outfield directions (degrees from north):
- Center field: ~45° (NE)
- Left-center field: ~25° (NNE)
- Right-center field: ~65° (ENE)

**Important**: Weather APIs report wind direction as the direction wind blows **FROM**. We must convert to the direction it blows **TO** before projecting.

In [5]:
# Merge weather into game data
games_full = games.merge(weather_df, on='game_pk', how='left')

# --- Wind direction bucketing ---
def bucket_wind_dir(deg):
    """Bucket wind direction (degrees) into 8 compass directions."""
    if pd.isna(deg):
        return np.nan
    buckets = ['N', 'NE', 'E', 'SE', 'S', 'SW', 'W', 'NW']
    idx = int(((deg + 22.5) % 360) / 45)
    return buckets[idx]

games_full['wind_dir_bucket'] = games_full['wdir'].apply(bucket_wind_dir)

# --- Wind projections onto outfield vectors ---
def compute_wind_projection(wdir, wspd, outfield_dir):
    """
    Project wind onto an outfield direction vector.
    
    wdir: direction wind blows FROM (meteorological convention, degrees)
    wspd: wind speed (km/h)
    outfield_dir: compass bearing from home plate to outfield (degrees from north)
    
    Returns: positive = blowing OUT toward outfield, negative = blowing IN
    """
    if pd.isna(wdir) or pd.isna(wspd):
        return np.nan
    # Wind blows FROM wdir, so it travels TOWARD (wdir + 180)
    wind_toward = (wdir + 180) % 360
    # Project onto outfield direction
    angle_diff = wind_toward - outfield_dir
    return wspd * np.cos(np.radians(angle_diff))

games_full['wind_cf'] = games_full.apply(
    lambda r: compute_wind_projection(r['wdir'], r['wspd'], CF_DIR), axis=1
)
games_full['wind_lcf'] = games_full.apply(
    lambda r: compute_wind_projection(r['wdir'], r['wspd'], LCF_DIR), axis=1
)
games_full['wind_rcf'] = games_full.apply(
    lambda r: compute_wind_projection(r['wdir'], r['wspd'], RCF_DIR), axis=1
)

print("Wind projection summary (positive = blowing out, negative = blowing in):")
print(games_full[['wind_cf', 'wind_lcf', 'wind_rcf']].describe())

Wind projection summary (positive = blowing out, negative = blowing in):
          wind_cf    wind_lcf    wind_rcf
count  758.000000  758.000000  758.000000
mean     3.193065    3.307969    2.693031
std     10.419796    9.947028   10.505853
min    -30.264782  -28.474915  -31.712906
25%     -4.002775   -3.449310   -4.794031
50%      3.916227    3.907474    4.283817
75%     11.224834   10.685184   10.666014
max     27.999528   28.631911   28.267416


## Section 4: Final Assembly

Convert units, order columns, and round to sensible precision.

In [6]:
# Unit conversions
games_full['temp_f'] = games_full['temp_c'] * 9/5 + 32
games_full['wspd_mph'] = games_full['wspd'] * 0.621371

# Final column order
final_columns = [
    # Game identification
    'game_pk', 'game_date', 'season', 'away_team', 'game_start', 'start_hour',
    # Scoring
    'home_runs_scored', 'away_runs_scored', 'total_runs',
    # Batting stats (both teams combined)
    'home_runs_hit', 'strikeouts', 'walks', 'hits',
    'total_pitches', 'avg_exit_velocity',
    'n_barrels', 'n_bbe', 'barrel_rate', 'hr_h_ratio',
    # Weather
    'temp_f', 'temp_c', 'rhum', 'pres', 'prcp',
    'wspd', 'wspd_mph', 'wdir', 'wind_dir_bucket',
    # Wind projections
    'wind_cf', 'wind_lcf', 'wind_rcf',
]

fenway_data = games_full[final_columns].copy()
fenway_data = fenway_data.sort_values('game_date').reset_index(drop=True)

# Round floating point columns
round_map = {
    'avg_exit_velocity': 1, 'barrel_rate': 4, 'hr_h_ratio': 4,
    'temp_f': 1, 'temp_c': 1, 'rhum': 1, 'pres': 1, 'prcp': 2,
    'wspd': 1, 'wspd_mph': 1, 'wdir': 1,
    'wind_cf': 2, 'wind_lcf': 2, 'wind_rcf': 2,
}
for col, decimals in round_map.items():
    fenway_data[col] = fenway_data[col].round(decimals)

print(f"Final dataset: {fenway_data.shape[0]} rows x {fenway_data.shape[1]} columns")

Final dataset: 758 rows x 31 columns


## Section 5: Validation

Verify row counts per season, check for nulls, and sanity-check summary statistics.

In [7]:
print("=" * 70)
print("VALIDATION REPORT")
print("=" * 70)

# 1. Row counts per season
print("\n--- Games per Season ---")
season_counts = fenway_data.groupby('season').size()
for year, count in season_counts.items():
    if year == 2020:
        expected = (25, 35)  # COVID shortened season
    else:
        expected = (75, 100)  # Normal: ~81 home games (wider range for doubleheaders)
    status = "OK" if expected[0] <= count <= expected[1] else "WARNING"
    print(f"  {year}: {count} games [{status}] (expected {expected[0]}-{expected[1]})")
print(f"  TOTAL: {len(fenway_data)} games")

# 2. Null check
print("\n--- Null Counts ---")
key_cols = ['total_runs', 'home_runs_hit', 'strikeouts', 'walks',
            'total_pitches', 'avg_exit_velocity', 'barrel_rate',
            'temp_f', 'wspd', 'wdir', 'wind_cf', 'game_start']
for col in key_cols:
    n_null = fenway_data[col].isna().sum()
    pct = 100 * n_null / len(fenway_data)
    status = "OK" if pct < 5 else "WARNING"
    print(f"  {col}: {n_null} nulls ({pct:.1f}%) [{status}]")

# 3. Summary statistics sanity checks
print("\n--- Sanity Checks ---")
checks = [
    ('Avg total runs/game', fenway_data['total_runs'].mean(), '~8-10'),
    ('Avg HR/game', fenway_data['home_runs_hit'].mean(), '~2-3'),
    ('Avg K/game', fenway_data['strikeouts'].mean(), '~16-18'),
    ('Avg BB/game', fenway_data['walks'].mean(), '~6-7'),
    ('Avg exit velocity', fenway_data['avg_exit_velocity'].mean(), '~87-89 mph'),
    ('Avg barrel rate', fenway_data['barrel_rate'].mean(), '~0.06-0.08'),
    ('Avg game temp', fenway_data['temp_f'].mean(), '~60-70 F'),
    ('Min game temp', fenway_data['temp_f'].min(), '>30 F'),
    ('Max game temp', fenway_data['temp_f'].max(), '<105 F'),
    ('Avg wind speed (km/h)', fenway_data['wspd'].mean(), '~10-20 km/h'),
]
for label, val, expected in checks:
    print(f"  {label}: {val:.2f} (expected {expected})")

# 4. Full summary statistics
print("\n--- Summary Statistics ---")
print(fenway_data.describe().T[['mean', 'std', 'min', 'max']].to_string())

VALIDATION REPORT

--- Games per Season ---
  2016: 80 games [OK] (expected 75-100)
  2017: 81 games [OK] (expected 75-100)
  2018: 81 games [OK] (expected 75-100)
  2019: 81 games [OK] (expected 75-100)
  2020: 30 games [OK] (expected 25-35)
  2021: 81 games [OK] (expected 75-100)
  2022: 81 games [OK] (expected 75-100)
  2023: 81 games [OK] (expected 75-100)
  2024: 81 games [OK] (expected 75-100)
  2025: 81 games [OK] (expected 75-100)
  TOTAL: 758 games

--- Null Counts ---
  total_runs: 0 nulls (0.0%) [OK]
  home_runs_hit: 0 nulls (0.0%) [OK]
  strikeouts: 0 nulls (0.0%) [OK]
  walks: 0 nulls (0.0%) [OK]
  total_pitches: 0 nulls (0.0%) [OK]
  avg_exit_velocity: 0 nulls (0.0%) [OK]
  barrel_rate: 0 nulls (0.0%) [OK]
  temp_f: 0 nulls (0.0%) [OK]
  wspd: 0 nulls (0.0%) [OK]
  wdir: 0 nulls (0.0%) [OK]
  wind_cf: 0 nulls (0.0%) [OK]
  game_start: 0 nulls (0.0%) [OK]

--- Sanity Checks ---
  Avg total runs/game: 10.05 (expected ~8-10)
  Avg HR/game: 2.32 (expected ~2-3)
  Avg K/game: 

In [8]:
# Print dataset header for inspection
print("\n--- First 10 Rows ---")
fenway_data.head(10)


--- First 10 Rows ---


,game_pk,game_date,season,away_team,game_start,start_hour,home_runs_scored,away_runs_scored,total_runs,home_runs_hit,strikeouts,walks,hits,total_pitches,avg_exit_velocity,n_barrels,n_bbe,barrel_rate,hr_h_ratio,temp_f,temp_c,rhum,pres,prcp,wspd,wspd_mph,wdir,wind_dir_bucket,wind_cf,wind_lcf,wind_rcf
0,446958,2016-04-11,2016,BAL,2016-04-11 14:05:00,14,7,9,16,3,23,8,20,330,84.0,6,49,0.1224,0.1500,55.9,13.3,52.0,1018.0,0.0,27.1,16.8,196.3,S,23.75,26.76,17.88
1,446970,2016-04-12,2016,BAL,2016-04-12 19:10:00,19,5,9,14,4,13,6,21,332,83.7,3,59,0.0508,0.1905,44.1,6.7,72.7,1021.2,0.0,16.9,10.5,311.3,NW,1.08,-4.76,6.80
2,446983,2016-04-13,2016,BAL,2016-04-13 19:10:00,19,4,2,6,1,17,9,17,310,83.6,4,47,0.0851,0.0588,40.0,4.5,75.0,1026.9,0.0,7.4,4.6,129.0,SE,-0.77,1.78,-3.23
3,447020,2016-04-15,2016,TOR,2016-04-15 19:10:00,19,5,3,8,2,19,8,12,312,83.7,6,42,0.1429,0.1667,44.2,6.8,59.0,1028.9,0.0,12.8,7.9,47.0,NE,-12.76,-11.84,-12.14
4,447035,2016-04-16,2016,TOR,2016-04-16 16:05:00,16,4,2,6,1,15,2,16,253,81.2,2,48,0.0417,0.0625,47.4,8.5,58.3,1030.5,0.0,14.0,8.7,68.7,E,-12.85,-10.15,-14.00
5,447050,2016-04-17,2016,TOR,2016-04-17 13:35:00,13,3,5,8,2,22,5,18,312,83.5,2,49,0.0408,0.1111,58.5,14.7,42.3,1026.5,0.0,8.5,5.3,77.7,E,-7.13,-5.14,-8.26
6,447057,2016-04-18,2016,TOR,2016-04-18 11:05:00,11,3,4,7,0,11,8,15,296,85.3,3,51,0.0588,0.0000,60.3,15.7,51.7,1022.1,0.0,15.8,9.8,61.7,NE,-15.14,-12.67,-15.77
7,447071,2016-04-19,2016,TB,2016-04-19 19:10:00,19,0,3,3,1,27,7,7,305,81.6,3,40,0.0750,0.1429,49.0,9.4,47.7,1016.4,0.0,5.4,3.4,355.3,N,-3.49,-4.69,-1.87
8,447086,2016-04-20,2016,TB,2016-04-20 19:10:00,19,7,3,10,2,20,5,19,283,85.8,5,48,0.1042,0.1053,45.6,7.5,69.7,1022.2,0.0,15.5,9.6,191.0,S,12.85,15.04,9.12
9,447100,2016-04-21,2016,TB,2016-04-21 13:35:00,13,8,12,20,4,16,11,30,374,84.9,7,67,0.1045,0.1333,70.4,21.3,30.3,1013.7,0.0,14.0,8.7,227.3,SW,13.96,12.92,13.31


## Section 6: Save to CSV

In [9]:
# Save final dataset
output_dir = os.path.join('..', 'Final Datasets')
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(output_dir, OUTPUT_FILE)
fenway_data.to_csv(output_path, index=False)

print(f"Saved to: {os.path.abspath(output_path)}")
print(f"File size: {os.path.getsize(output_path) / 1024:.1f} KB")
print(f"Rows: {len(fenway_data)}, Columns: {len(fenway_data.columns)}")

# Verify roundtrip
verify = pd.read_csv(output_path)
assert verify.shape == fenway_data.shape, f"Shape mismatch: {verify.shape} vs {fenway_data.shape}"
print("\nSave & reload verification: PASSED")

Saved to: /Users/avabrown/Desktop/DATASCI 192A/Stadium Datasets/Final Datasets/fenway_data_2016.csv
File size: 114.0 KB
Rows: 758, Columns: 31

Save & reload verification: PASSED
